# 03 - AQUAVIEW data via STAC (Python)

**File:** `notebooks/03_aquaview_stac_python.ipynb`

**What this does:** Searches the public AQUAVIEW catalogue, picks a glider deployment, downloads a slice of it into a table and plots it.

**How to run it:** Open this file in JupyterLab, check that the kernel shown in the
top-right corner says **Python 3**, then choose *Run > Run All Cells*.

**Inputs:** none -- reads from the public AQUAVIEW API over the internet

**Outputs:** a table of glider measurements and a temperature-vs-depth plot

## What is STAC?

AQUAVIEW publishes its catalogue as a **STAC** API (SpatioTemporal Asset
Catalog) -- a shared standard for describing data that has a location and a
time. The useful part: it is ordinary JSON over ordinary web requests, so no
special library is needed.

Three ideas cover almost everything:

| Term | Meaning |
|---|---|
| **Collection** | A group of related data, e.g. `IOOS` = the IOOS Glider Data Assembly Center. |
| **Item** | One thing inside a collection, e.g. a single glider deployment. |
| **Asset** | A downloadable file attached to an item, e.g. that deployment as CSV. |

The catalogue is **public** -- no account, no API key, no login.

In [ ]:
import requests

# The AQUAVIEW catalogue. No key or login needed.
BASE = "https://service.aquaview.org/stac"

response = requests.get(f"{BASE}/collections", timeout=60)
response.raise_for_status()      # stop here with a clear error if the request failed

collections = response.json()["collections"]
print(f"{len(collections)} collections available. The first 10:\n")

for collection in collections[:10]:
    print(f"  {collection['id']:16s} {collection.get('title', '')[:58]}")

## 2. Searching for items

`IOOS` is the IOOS Glider Data Assembly Center -- autonomous underwater gliders.
Let us look around **Hawaiʻi** -- the same waters the hackathon glider data
comes from, so the two fit together.

In [ ]:
search = {
    "collections": "IOOS",
    "bbox": "-161,18,-154,23",   # west, south, east, north
    "limit": 5,
}

response = requests.get(f"{BASE}/search", params=search, timeout=60)
response.raise_for_status()
results = response.json()

print("items matching the search:", results["numberMatched"])
print("returned in this page:    ", len(results["features"]))
print()

for feature in results["features"]:
    print(" ", feature["id"])

### Reading the search settings

- `collections` -- which collection to look in. Leave it out to search all 100.
- `bbox` -- a box on the map, given as `west,south,east,north`. The values above
  cover the Hawaiian Islands. The Glider Rodeo waypoints sit just off Oʻahu,
  around 21.2 N, 158.2 W, well inside that box.
- `limit` -- how many items to return at once.

You can also pass `datetime` to restrict the time range, like
`2016-01-01T00:00:00Z/2017-12-31T23:59:59Z`.

> **Watch the order in `bbox`.** It is longitude first, then latitude
> (`west,south,east,north`). Getting this backwards is the most common mistake,
> and it usually returns nothing at all rather than an error.

## 3. Looking at one item

In [ ]:
item = results["features"][0]

print("id:        ", item["id"])
print("collection:", item["collection"])

# ERDDAP offers the same data in many formats, so just count them and
# show a few rather than printing all of them.
print(f"assets:     {len(item['assets'])} formats available")
print("            e.g.", ", ".join(sorted(item["assets"])[:10]))

### From item to actual data

The item above is only a *description*. The real data lives at one of the URLs
in `assets`. We want the `csv` one.

These particular CSV links point at **ERDDAP**, a data server widely used in
ocean science. Two practical things follow:

> **Always ask for a time range.** A full glider deployment can be hundreds of
> megabytes, and without a time limit you download all of it. Adding
> `?&time>=...&time<=...` to the URL asks the server to send just that slice.

For these glider items the deployment start date is written into the item id
(`sg626-`**`20250729`**`T1452`), so the cell below pulls the date out of the
id and asks for that first day -- a day we know has data in it.

In [ ]:
import re

csv_url = item["assets"]["csv"]["href"]

# Pull the deployment start date out of the item id, e.g. "...-20250729T1452".
day = re.search(r"-(\d{8})T", item["id"]).group(1)
start = f"{day[:4]}-{day[4:6]}-{day[6:]}"

data_url = f"{csv_url}?&time>={start}&time<={start}T23:59:59Z"

print("deployment started:", start)
print("downloading:", data_url)

### One quirk: ERDDAP sends two header rows

An ERDDAP CSV has the column *names* on line 1 and the column *units* on line 2,
with the data starting on line 3. That second line has to be skipped, or every
number is read as text and no arithmetic works.

In [ ]:
import pandas as pd

# skiprows=[1] drops the units row, keeping the names on line 1.
glider = pd.read_csv(data_url, skiprows=[1])

print(f"{len(glider)} measurements, {len(glider.columns)} columns")
glider[["time", "latitude", "longitude", "depth", "temperature"]].head()

## 4. A quick look at the data

A glider dives up and down as it travels, so plotting temperature against depth
shows the shape of the water column. Depth increases downwards, so the axis is
flipped to match how we picture the ocean.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(5, 6))
ax.scatter(glider["temperature"], glider["depth"], s=4, alpha=0.4)

ax.invert_yaxis()
ax.set_xlabel("Temperature (C)")
ax.set_ylabel("Depth (m)")
ax.set_title(f"{item['id']}\n{start}")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Done

Next: **`04_glider_satellite_python.ipynb`**, a longer real-world workflow comparing a glider
track against satellite data.

After that, start building your own thing -- see the [main README](../README.md).